In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.animation as animation
from IPython.display import HTML

# 환경 생성
env = gym.make("FrozenLake-v1", is_slippery=False)

# Q-table 초기화 (상태 수 × 행동 수)
q_table = np.zeros((env.observation_space.n, env.action_space.n))

# 학습 설정
episodes = 1000  # 최대 에피소드 수
epsilon = 0.1    # ε-greedy 탐험 확률
episode_paths = []          # 각 에피소드별 상태 이동 경로 저장
episode_text_paths = []     # 각 에피소드별 행동-결과 경로 저장

first_success_episode = None    # 첫 성공 기록용 변수
success_count = 0               # 에피소드 성공 횟수 카운트용 변수

# 결과를 직관적으로 보기 위해 행동 번호를 방향 기호로 매핑
action_names = ["←", "↓", "→", "↑"]

# Q-learning 학습 루프
for episode in range(episodes):
    state, _ = env.reset()
    done = False
    episode_path = []
    episode_text_path = []

    while not done:
        # ε-greedy 탐험: 무작위 vs 최적행동
        if np.random.rand() < epsilon or np.sum(q_table[state]) == 0:
            action = env.action_space.sample()  # 탐험(무작위 행동)
        else:
            max_actions = np.flatnonzero(q_table[state] == np.max(q_table[state]))
            action = np.random.choice(max_actions)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        q_table[state, action] = reward + np.max(q_table[next_state])

        # '행동-결과' 로그 저장
        move = action_names[action]
        if next_state == state:
            episode_text_path.append(f"{move}벽")
        else:
            episode_text_path.append(f"{move}{next_state}")

        episode_path.append((state, episode))
        state = next_state

    episode_path.append((state, episode))
    episode_paths.append(episode_path)
    episode_text_paths.append(episode_text_path)

    if reward == 1:
        success_count += 1

    if reward == 1 and first_success_episode is None:
        first_success_episode = episode
        print(f"\n 최초 성공 에피소드(에피소드 번호: {first_success_episode})의 '행동-결과' 경로:")
        print("  ▶ ".join(episode_text_path))

    # ε 점차 감소 (decaying ε-greedy)
    epsilon = max(0.01, epsilon * 0.995)  # 0.995는 감쇠율, 0.01은 최소 ε

# ---------------------------------------------------------------
# 최종 에피소드의 '행동-결과' 로그 출력
print("\n 최종 에피소드의 '행동-결과' 경로:")
print("  ▶ ".join(episode_text_paths[-1]))

# ---------------------------------------------------------------
# 전체 에피소드 성공 횟수 및 성공률 출력
print(f"\n총 에피소드 중 성공 횟수: {success_count}회 / {episodes}회")
print(f"성공률: {success_count / episodes * 100:.2f}%")

# ---------------------------------------------------------------
# 최종 결과 출력
print("\n=== Final Q-table (최종 Q값) ===")
print(q_table.astype(int))
print()

# ---------------------------------------------------------------
# 시각화를 위한 에피소드 구간 선택: 초기 3개, 최초 성공 후 20개, 마지막 3개
selected_trajectories = []
selected_trajectories.extend(episode_paths[0:3])
selected_trajectories.extend(episode_paths[first_success_episode:first_success_episode+20])
selected_trajectories.extend(episode_paths[-3:])
# 선택된 에피소드들을 일렬로 펼쳐(flatten) 리스트로 저장
flat_trajectory = [state for ep in selected_trajectories for state in ep]

# ---------------------------------------------------------------
# 애니메이션 함수 (FrozenLake 에이전트 경로 시각화)
def animate_agent(trajectory, size=4):
    grid = np.array(['S', 'F', 'F', 'F',
                     'F', 'H', 'F', 'H',
                     'F', 'F', 'F', 'H',
                     'H', 'F', 'F', 'G']).reshape((size, size))
    color_values = np.where(grid == 'S', 0.3,
                    np.where(grid == 'F', 0.1,
                    np.where(grid == 'H', 0.8, 0.5)))

    fig, ax = plt.subplots(figsize=(5, 5))
    sns.heatmap(color_values, cbar=False, annot=grid, fmt='s',
                cmap='Blues', linewidths=1, linecolor='gray',
                square=True, ax=ax, annot_kws={"fontsize":12})

    colors = ['orange', 'red', 'blue', 'green', 'purple', 'black', 'magenta']
    agent_marker, = ax.plot([], [], marker="o", color="red", markersize=30)

    ax.set_title('FrozenLake Map', fontsize=14)

    def update(frame):
        state, ep_id = trajectory[frame]
        row, col = divmod(state, size)
        agent_marker.set_color(colors[ep_id % len(colors)])
        agent_marker.set_data([col + 0.5], [row + 0.5])
        return agent_marker,

    ani = animation.FuncAnimation(fig, update, frames=len(trajectory), blit=True, repeat=False, interval=300)
    plt.close()
    return ani

# 애니메이션 실행
#HTML(animate_agent(flat_trajectory).to_jshtml())


 최초 성공 에피소드(에피소드 번호: 73)의 '행동-결과' 경로:
↓4  ▶ ↑0  ▶ ↑벽  ▶ ←벽  ▶ ←벽  ▶ ↓4  ▶ ↓8  ▶ ↑4  ▶ ←벽  ▶ ↑0  ▶ →1  ▶ ↑벽  ▶ →2  ▶ ↑벽  ▶ →3  ▶ ←2  ▶ ←1  ▶ →2  ▶ ↓6  ▶ ↑2  ▶ ↓6  ▶ ↓10  ▶ ↓14  ▶ →15

 최종 에피소드의 '행동-결과' 경로:
←벽  ▶ →1  ▶ ↑벽  ▶ ↑벽  ▶ →2  ▶ →3  ▶ →벽  ▶ →벽  ▶ ←2  ▶ ↑벽  ▶ ←1  ▶ ←0  ▶ ←벽  ▶ ←벽  ▶ →1  ▶ ←0  ▶ ↓4  ▶ ↓8  ▶ ↑4  ▶ ←벽  ▶ ←벽  ▶ ←벽  ▶ ↑0  ▶ →1  ▶ ←0  ▶ ↑벽  ▶ ←벽  ▶ ←벽  ▶ →1  ▶ ↑벽  ▶ ↑벽  ▶ ←0  ▶ ↑벽  ▶ ←벽  ▶ ←벽  ▶ ↓4  ▶ ↓8  ▶ →9  ▶ ←8  ▶ ←벽  ▶ ←벽  ▶ ←벽  ▶ ←벽  ▶ →9  ▶ ←8  ▶ ←벽  ▶ ←벽  ▶ ←벽  ▶ →9  ▶ ↓13  ▶ ↑9  ▶ ←8  ▶ ←벽  ▶ →9  ▶ ←8  ▶ →9  ▶ ←8  ▶ →9  ▶ ↓13  ▶ →14  ▶ ↓벽  ▶ ↓벽  ▶ ↓벽  ▶ ↓벽  ▶ →15

총 에피소드 중 성공 횟수: 614회 / 1000회
성공률: 61.40%

=== Final Q-table (최종 Q값) ===
[[1 1 1 1]
 [1 0 1 1]
 [1 1 1 1]
 [1 0 1 1]
 [1 1 0 1]
 [0 0 0 0]
 [0 1 0 1]
 [0 0 0 0]
 [1 0 1 1]
 [1 1 1 0]
 [1 1 0 1]
 [0 0 0 0]
 [0 0 0 0]
 [0 1 1 1]
 [1 1 1 1]
 [0 0 0 0]]

